
# Practice Assignment

# Feature Engineering, Machine Learning Development and Ray

---

## Learning Outcomes

After completing this notebook students should be able to:

1. Handle missing values
2. Remove duplicates
3. Detect outliers
4. Perform scaling
5. Perform normalization
6. Encode categorical variables
7. Create derived features
8. Create temporal features
9. Build Scikit-Learn Pipelines
10. Understand Ray Architecture
11. Create Remote Functions
12. Use Ray Tasks
13. Use Actors
14. Work with Ray Data
15. Build Ray Train workflows
16. Perform Hyperparameter Tuning with Ray Tune
17. Deploy Models using Ray Serve

---

### Instructions

- Attempt all TODO sections first.
- Use hints when needed.
- View solutions only after attempting.



# Exercise 1: Install Required Libraries

## Background


A Machine Learning Engineer is setting up a new
development environment.

Required libraries must be installed before
building ML pipelines.


## Dataset / Input


Libraries

1. pandas
2. scikit-learn
3. ray


## Task


Install the required libraries.


## Hint


Use pip install.



In [1]:
import subprocess
import sys

# Install libraries
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn", "ray"])
print("Libraries installed successfully!")


Libraries installed successfully!


Background

A telecom company collected customer information from multiple sources.
Some records have missing values because customers skipped fields during registration.

Dataset / Input

The DataFrame df contains:

| age | monthly_bill | city |
|------|------|------|
| 25 | 300 | Chennai |
| 30 | 500 | Delhi |
| NaN | 450 | Delhi |
| 40 | NaN | Mumbai |
| 35 | 700 | Chennai |

Task

Replace missing values as follows:

1. Replace missing values in age using the mean of the age column.
2. Replace missing values in monthly_bill using the mean of the monthly_bill column.
3. Store the updated values back into df.

Expected Result

The DataFrame should contain no missing values in age and monthly_bill.


# Exercise 2: Import Libraries

## Background


Most ML workflows begin by importing
required libraries.


## Dataset / Input


Libraries

pandas
numpy
ray


## Task


Import required libraries.


## Hint


Use import statements.



In [2]:
# Import libraries
import pandas as pd
import numpy as np
import ray
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn import datasets

print("Libraries imported successfully!")


Libraries imported successfully!



# Feature Engineering

In this section we will build a customer
analytics dataset and perform feature
engineering operations.



# Exercise 3: Create Customer Dataset

## Background


A telecom company stores customer records.

The dataset will be reused throughout
the notebook.


## Dataset / Input


Customer Information


## Task


Create a Pandas DataFrame.


## Hint


Use pd.DataFrame().



In [3]:
# Create DataFrame
df = pd.DataFrame({
    'age': [25, 30, np.nan, 40, 35, 28, np.nan, 45],
    'monthly_bill': [300, 500, 450, np.nan, 700, 600, 550, 800],
    'city': ['Chennai', 'Delhi', 'Delhi', 'Mumbai', 'Chennai', 'Bangalore', 'Mumbai', 'Bangalore'],
    'transaction_date': pd.date_range('2024-01-01', periods=8, freq='D')
})

print("DataFrame created:")
print(df)
print("\nDataFrame Info:")
print(df.info())


DataFrame created:
    age  monthly_bill       city transaction_date
0  25.0         300.0    Chennai       2024-01-01
1  30.0         500.0      Delhi       2024-01-02
2   NaN         450.0      Delhi       2024-01-03
3  40.0           NaN     Mumbai       2024-01-04
4  35.0         700.0    Chennai       2024-01-05
5  28.0         600.0  Bangalore       2024-01-06
6   NaN         550.0     Mumbai       2024-01-07
7  45.0         800.0  Bangalore       2024-01-08

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   age               6 non-null      float64       
 1   monthly_bill      7 non-null      float64       
 2   city              8 non-null      object        
 3   transaction_date  8 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 388.0+ bytes
None



# Exercise 4: Handle Missing Values

## Background


Some customer records contain missing values.

Models cannot train reliably with missing data.


## Dataset / Input


Customer Dataset


## Task


Replace missing values using mean imputation.


## Hint


Use fillna().



In [ ]:
# Handle missing values
df['age'] = df['age'].fillna(df['age'].mean())
df['monthly_bill'] = df['monthly_bill'].fillna(df['monthly_bill'].mean())

print("Missing values handled:")
print(df)
print("\nMissing values count:")
print(df.isnull().sum())


Missing values handled:
         age  monthly_bill       city transaction_date
0  25.000000    300.000000    Chennai       2024-01-01
1  30.000000    500.000000      Delhi       2024-01-02
2  33.833333    450.000000      Delhi       2024-01-03
3  40.000000    557.142857     Mumbai       2024-01-04
4  35.000000    700.000000    Chennai       2024-01-05
5  28.000000    600.000000  Bangalore       2024-01-06
6  33.833333    550.000000     Mumbai       2024-01-07
7  45.000000    800.000000  Bangalore       2024-01-08

Missing values count:
age                 0
monthly_bill        0
city                0
transaction_date    0
dtype: int64



# Exercise 5: Remove Duplicates

## Background


Duplicate records may distort analysis.


## Dataset / Input


Customer Dataset


## Task


Remove duplicate rows.


## Hint


Use drop_duplicates().



In [ ]:
# Remove duplicates
df_no_dup = df.drop_duplicates()

print("Duplicates removed:")
print(f"Original shape: {df.shape}, After removing duplicates: {df_no_dup.shape}")
print(df_no_dup)


Duplicates removed:
Original shape: (8, 4), After removing duplicates: (8, 4)
         age  monthly_bill       city transaction_date
0  25.000000    300.000000    Chennai       2024-01-01
1  30.000000    500.000000      Delhi       2024-01-02
2  33.833333    450.000000      Delhi       2024-01-03
3  40.000000    557.142857     Mumbai       2024-01-04
4  35.000000    700.000000    Chennai       2024-01-05
5  28.000000    600.000000  Bangalore       2024-01-06
6  33.833333    550.000000     Mumbai       2024-01-07
7  45.000000    800.000000  Bangalore       2024-01-08



# Exercise 6: Detect Outliers

## Background


Management suspects unusually high bills.


## Dataset / Input


monthly_bill column


## Task


Identify outliers using IQR.


## Hint


Use quantiles.



In [ ]:
# Detect outliers using IQR
Q1 = df['monthly_bill'].quantile(0.25)
Q3 = df['monthly_bill'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['monthly_bill'] < lower_bound) | (df['monthly_bill'] > upper_bound)]

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")
print("\nOutliers detected:")
print(outliers)


Q1: 487.5, Q3: 625.0, IQR: 137.5
Lower bound: 281.25, Upper bound: 831.25

Outliers detected:
Empty DataFrame
Columns: [age, monthly_bill, city, transaction_date]
Index: []



# Exercise 7: Standard Scaling

## Background


Features should often be scaled before
model training.


## Dataset / Input


age
monthly_bill


## Task


Apply StandardScaler.


## Hint


Use StandardScaler.



In [ ]:
# Scale features using StandardScaler
scaler = StandardScaler()
features_to_scale = ['age', 'monthly_bill']
df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Scaled features (StandardScaler):")
print(df_scaled[features_to_scale])
print("\nMean after scaling:", df_scaled[features_to_scale].mean())
print("Std after scaling:", df_scaled[features_to_scale].std())


Scaled features (StandardScaler):
        age  monthly_bill
0 -1.465033     -1.806220
1 -0.635769     -0.401382
2  0.000000     -0.752591
3  1.022759      0.000000
4  0.193495      1.003455
5 -0.967475      0.301037
6  0.000000     -0.050173
7  1.852023      1.705874

Mean after scaling: age            -2.775558e-16
monthly_bill    2.220446e-16
dtype: float64
Std after scaling: age             1.069045
monthly_bill    1.069045
dtype: float64



# Exercise 8: Min Max Normalization

## Background


Some algorithms require values between
0 and 1.


## Dataset / Input


age
monthly_bill


## Task


Apply MinMaxScaler.


## Hint


Use MinMaxScaler.



In [ ]:
# Normalize features using MinMaxScaler
minmax_scaler = MinMaxScaler()
features_to_normalize = ['age', 'monthly_bill']
df_normalized = df.copy()
df_normalized[features_to_normalize] = minmax_scaler.fit_transform(df[features_to_normalize])

print("Normalized features (MinMaxScaler - range [0, 1]):")
print(df_normalized[features_to_normalize])
print("\nMin after normalization:", df_normalized[features_to_normalize].min())
print("Max after normalization:", df_normalized[features_to_normalize].max())


Normalized features (MinMaxScaler - range [0, 1]):
        age  monthly_bill
0  0.000000      0.000000
1  0.250000      0.400000
2  0.441667      0.300000
3  0.750000      0.514286
4  0.500000      0.800000
5  0.150000      0.600000
6  0.441667      0.500000
7  1.000000      1.000000

Min after normalization: age             0.0
monthly_bill    0.0
dtype: float64
Max after normalization: age             1.0
monthly_bill    1.0
dtype: float64



# Exercise 9: Label Encoding

## Background


Machine learning models require
numerical features.


## Dataset / Input


city column


## Task


Encode city values.


## Hint


Use LabelEncoder.



In [ ]:
# Label encode city column
label_encoder = LabelEncoder()
df_encoded = df.copy()
df_encoded['city_encoded'] = label_encoder.fit_transform(df['city'])

print("Label encoded city column:")
print(df_encoded[['city', 'city_encoded']])
print("\nLabel mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")


Label encoded city column:
        city  city_encoded
0    Chennai             1
1      Delhi             2
2      Delhi             2
3     Mumbai             3
4    Chennai             1
5  Bangalore             0
6     Mumbai             3
7  Bangalore             0

Label mapping:
Bangalore -> 0
Chennai -> 1
Delhi -> 2
Mumbai -> 3



# Exercise 10: One Hot Encoding

## Background


Categorical variables can also be
represented using one-hot encoding.


## Dataset / Input


city column


## Task


Create one-hot encoded features.


## Hint


Use get_dummies().



In [ ]:
# One hot encode city column
df_onehot = pd.get_dummies(df, columns=['city'], prefix='city')

print("One-hot encoded city column:")
print(df_onehot.columns)
print(df_onehot.head())


One-hot encoded city column:
Index(['age', 'monthly_bill', 'transaction_date', 'city_Bangalore',
       'city_Chennai', 'city_Delhi', 'city_Mumbai'],
      dtype='object')
         age  monthly_bill transaction_date  city_Bangalore  city_Chennai  \
0  25.000000    300.000000       2024-01-01           False          True   
1  30.000000    500.000000       2024-01-02           False         False   
2  33.833333    450.000000       2024-01-03           False         False   
3  40.000000    557.142857       2024-01-04           False         False   
4  35.000000    700.000000       2024-01-05           False          True   

   city_Delhi  city_Mumbai  
0       False        False  
1        True        False  
2        True        False  
3       False         True  
4       False        False  



# Exercise 11: Create Derived Features

## Background


Revenue analysts want yearly billing.


## Dataset / Input


monthly_bill


## Task


Create annual_bill feature.


## Hint


Multiply by 12.



In [ ]:
# Create derived feature - annual_bill
df_features = df.copy()
df_features['annual_bill'] = df_features['monthly_bill'] * 12

print("Derived feature (annual_bill):")
print(df_features[['monthly_bill', 'annual_bill']])


Derived feature (annual_bill):
   monthly_bill  annual_bill
0    300.000000  3600.000000
1    500.000000  6000.000000
2    450.000000  5400.000000
3    557.142857  6685.714286
4    700.000000  8400.000000
5    600.000000  7200.000000
6    550.000000  6600.000000
7    800.000000  9600.000000



# Exercise 12: Create Temporal Features

## Background


Customer transactions contain dates.


## Dataset / Input


transaction_date


## Task


Extract month feature.


## Hint


Use pd.to_datetime().



In [ ]:
# Extract temporal features
df_temporal = df.copy()
df_temporal['transaction_date'] = pd.to_datetime(df_temporal['transaction_date'])
df_temporal['month'] = df_temporal['transaction_date'].dt.month
df_temporal['day'] = df_temporal['transaction_date'].dt.day
df_temporal['dayofweek'] = df_temporal['transaction_date'].dt.dayofweek

print("Temporal features extracted:")
print(df_temporal[['transaction_date', 'month', 'day', 'dayofweek']])


Temporal features extracted:
  transaction_date  month  day  dayofweek
0       2024-01-01      1    1          0
1       2024-01-02      1    2          1
2       2024-01-03      1    3          2
3       2024-01-04      1    4          3
4       2024-01-05      1    5          4
5       2024-01-06      1    6          5
6       2024-01-07      1    7          6
7       2024-01-08      1    8          0



# Exercise 13: Train Validation Test Split

## Background


Models should be evaluated properly.


## Dataset / Input


Customer Dataset


## Task


Create train and test splits.


## Hint


Use train_test_split().



In [ ]:
# Train-test split
X = df[['age', 'monthly_bill']]
y = (df['monthly_bill'] > 500).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTrain set:\n{X_train}")
print(f"\nTest set:\n{X_test}")


Training set size: (5, 2)
Test set size: (3, 2)

Train set:
         age  monthly_bill
7  45.000000    800.000000
2  33.833333    450.000000
4  35.000000    700.000000
3  40.000000    557.142857
6  33.833333    550.000000

Test set:
    age  monthly_bill
1  30.0         500.0
5  28.0         600.0
0  25.0         300.0



# Exercise 14: Scikit Learn Pipeline

## Background


Preprocessing and model training should
be reproducible.


## Dataset / Input


Customer Dataset


## Task


Create a pipeline.


## Hint


Use Pipeline().



In [ ]:
# Create a pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

# Train the pipeline
pipeline.fit(X_train, y_train)

# Score on test set
score = pipeline.score(X_test, y_test)
print(f"Pipeline created and trained successfully!")
print(f"Test set accuracy: {score:.3f}")


Pipeline created and trained successfully!
Test set accuracy: 0.667



# Ray Core

In this section we will learn Ray
through practical exercises.



# Exercise 15: Initialize Ray

## Background


Ray must be initialized before use.


## Dataset / Input


No Dataset


## Task


Start a local Ray runtime.


## Hint


Use ray.init().



In [ ]:
# Initialize Ray
if not ray.is_initialized():
    ray.init(ignore_reinit_error=True)
    print("Ray initialized successfully!")
else:
    print("Ray is already initialized")

print(f"Ray dashboard: http://127.0.0.1:8265")


2026-08-16 08:42:41,609	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Ray initialized successfully!
Ray dashboard: http://127.0.0.1:8265



# Exercise 16: Create Remote Function

## Background


A company wants distributed execution.


## Dataset / Input


Numbers


## Task


Create a remote function.


## Hint


Use @ray.remote.



In [ ]:
# Create a remote function
@ray.remote
def add(x, y):
    """Remote function to add two numbers"""
    return x + y

@ray.remote
def square(x):
    """Remote function to square a number"""
    return x ** 2

print("Remote functions created successfully!")
print(f"add function: {add}")
print(f"square function: {square}")


Remote functions created successfully!
add function: <ray.remote_function.RemoteFunction object at 0x7b1d8c85ee40>
square function: <ray.remote_function.RemoteFunction object at 0x7b1d8917f750>



# Exercise 17: Parallel Tasks

## Background


Many tasks should run simultaneously.


## Dataset / Input


Numbers 1-10


## Task


Execute tasks in parallel.


## Hint


Use .remote().



In [ ]:
# Execute tasks in parallel
futures = []
for i in range(1, 11):
    # Submit tasks to Ray
    result = square.remote(i)
    futures.append(result)

# Get results
results = ray.get(futures)

print("Parallel task execution completed!")
print(f"Squared values from 1-10: {results}")


Parallel task execution completed!
Squared values from 1-10: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]



# Exercise 18: Using ray.wait

## Background


Some tasks finish before others.


## Dataset / Input


Tasks


## Task


Wait for first completed task.


## Hint


Use ray.wait().



In [ ]:
# Wait for tasks using ray.wait()
import time

futures = []
for i in range(1, 6):
    result = square.remote(i)
    futures.append(result)

# Wait for first completed task
ready, not_ready = ray.wait(futures, num_returns=1)

print(f"First completed task result: {ray.get(ready[0])}")
print(f"Number of remaining tasks: {len(not_ready)}")

# Wait for remaining tasks
remaining_results = ray.get(not_ready)
print(f"Remaining results: {remaining_results}")


First completed task result: 1
Number of remaining tasks: 4
Remaining results: [4, 9, 16, 25]



# Exercise 19: Create Actor

## Background


Some applications require state.


## Dataset / Input


Counter Example


## Task


Create an Actor.


## Hint


Use class + @ray.remote.



In [ ]:
# Create an Actor
@ray.remote
class Counter:
    """Actor to maintain counter state"""
    def __init__(self):
        self.count = 0
    
    def increment(self):
        self.count += 1
        return self.count
    
    def get_count(self):
        return self.count

# Create an instance of the actor
counter = Counter.remote()
print("Counter actor created successfully!")
print(f"Actor reference: {counter}")


Counter actor created successfully!
Actor reference: Actor(Counter, 6a6afbd148e439d14923dd5a01000000)



# Exercise 20: Stateful Actor

## Background


Actors maintain state between calls.


## Dataset / Input


Counter


## Task


Increment counter multiple times.


## Hint


Call actor methods repeatedly.



In [ ]:
# Increment counter multiple times
results = []
for i in range(5):
    result = counter.increment.remote()
    results.append(result)

# Get all results
increment_results = ray.get(results)
print(f"Counter incremented: {increment_results}")

# Get final count
final_count = ray.get(counter.get_count.remote())
print(f"Final counter value: {final_count}")


Counter incremented: [1, 2, 3, 4, 5]
Final counter value: 5



# Ray Data

Ray Data provides scalable datasets
for machine learning and analytics.



# Exercise 21: Create Ray Dataset

## Background


An analytics team wants to process
customer records using Ray Data.


## Dataset / Input


Customer Records


## Task


Create a Ray Dataset.


## Hint


Use ray.data.from_items().



In [ ]:
# Create a Ray Dataset
customers = [
    {"customer": "A", "age": 25, "monthly_bill": 300},
    {"customer": "B", "age": 30, "monthly_bill": 500},
    {"customer": "C", "age": 35, "monthly_bill": 700},
    {"customer": "D", "age": 40, "monthly_bill": 900},
    {"customer": "E", "age": 28, "monthly_bill": 600}
]

dataset = ray.data.from_items(customers)

print("Ray Dataset created successfully!")
print(f"Dataset: {dataset}")
print(f"Dataset count: {dataset.count()}")


2026-08-16 08:42:55,868	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.
2026-08-16 08:42:55,959	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_0_0 =======
2026-08-16 08:42:55,961	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-16 08:42:55,962	INFO logging_progress.py:227 -- Active & requested resources: 0/0 CPU, 0.0B/0.0B object store
2026-08-16 08:42:55,962	INFO logging_progress.py:192 -- ============================================


Ray Dataset created successfully!


2026-08-16 08:42:55,970	INFO streaming_executor.py:327 -- ✔️  Dataset dataset_0_0 execution finished in 0.00 seconds


Dataset: shape: (5, 3)
╭──────────┬───────┬──────────────╮
│ customer ┆ age   ┆ monthly_bill │
│ ---      ┆ ---   ┆ ---          │
│ string   ┆ int64 ┆ int64        │
╞══════════╪═══════╪══════════════╡
│ A        ┆ 25    ┆ 300          │
│ B        ┆ 30    ┆ 500          │
│ C        ┆ 35    ┆ 700          │
│ D        ┆ 40    ┆ 900          │
│ E        ┆ 28    ┆ 600          │
╰──────────┴───────┴──────────────╯
(Showing 5 of 5 rows)
Dataset count: 5



# Exercise 22: Map Transformation

## Background


Management wants yearly billing
for every customer.


## Dataset / Input


Customer Dataset


## Task


Create annual_bill.


## Hint


Use map().



In [ ]:
# Map transformation to create annual_bill
def add_annual_bill(row):
    row["annual_bill"] = row["monthly_bill"] * 12
    return row

dataset_with_annual = dataset.map(add_annual_bill)

print("Map transformation applied!")
print("First few records with annual_bill:")
dataset_with_annual.show(3)


2026-08-16 08:42:56,024	INFO dataset.py:3979 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-08-16 08:42:56,036	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-08-16_08-42-25_900245_1674369/logs/ray-data
2026-08-16 08:42:56,038	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> LimitOperator[limit=3] -> TaskPoolMapOperator[Map(add_annual_bill)]
2026-08-16 08:42:56,074	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[Map(add_annual_bill)]. The job may hang forever unless the cluster scales up.
2026-08-16 08:42:56,079	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_2_0 =======
2026-08-16 08:42:56,081	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-16 08:42:56,082	INFO logging_progress.py:227 -- Active & requested resources:

Map transformation applied!
First few records with annual_bill:


2026-08-16 08:42:57,389	INFO streaming_executor.py:327 -- ✔️  Dataset dataset_2_0 execution finished in 1.35 seconds


{'customer': 'A', 'age': 25, 'monthly_bill': 300, 'annual_bill': 3600}
{'customer': 'B', 'age': 30, 'monthly_bill': 500, 'annual_bill': 6000}
{'customer': 'C', 'age': 35, 'monthly_bill': 700, 'annual_bill': 8400}



# Exercise 23: Filter Records

## Background


Marketing wants customers whose
monthly bill exceeds 400.


## Dataset / Input


Customer Dataset


## Task


Filter records.


## Hint


Use filter().



In [ ]:
# Filter records where monthly_bill > 400
def filter_high_bill(row):
    return row["monthly_bill"] > 400

filtered_dataset = dataset.filter(filter_high_bill)

print("Filtered dataset (monthly_bill > 400):")
filtered_dataset.show()


/home/sohang/.local/bin/miniconda3/lib/python3.13/site-packages/ray/data/dataset.py:1716: UserWarning: Use 'expr' instead of 'fn' when possible for performant filters.
  warnings.warn(
2026-08-16 08:42:57,577	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_4_0. Full logs are in /tmp/ray/session_2026-08-16_08-42-25_900245_1674369/logs/ray-data
2026-08-16 08:42:57,578	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> TaskPoolMapOperator[Filter(filter_high_bill)] -> LimitOperator[limit=20]


Filtered dataset (monthly_bill > 400):


2026-08-16 08:42:57,600	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[Filter(filter_high_bill)]. The job may hang forever unless the cluster scales up.
2026-08-16 08:42:57,617	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-08-16 08:42:57,618	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-16 08:42:57,619	INFO logging_progress.py:227 -- Active & requested resources: 0/0 CPU, 0.0B/0.0B object store
2026-08-16 08:42:57,621	INFO logging_progress.py:181 -- 
2026-08-16 08:42:57,622	INFO logging_progress.py:231 -- Filter(filter_high_bill): 0/1
2026-08-16 08:42:57,623	INFO logging_progress.py:233 --   Tasks: 1 [backpressured:tasks(ResourceBudget)]; Actors: 0; Queued blocks: 4 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-08-16 08:42:57,624	INFO logging_progress.py:231 -- limit=20: 0/1
2026-08-16 08:42:57,625	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (

{'customer': 'B', 'age': 30, 'monthly_bill': 500}
{'customer': 'C', 'age': 35, 'monthly_bill': 700}
{'customer': 'D', 'age': 40, 'monthly_bill': 900}
{'customer': 'E', 'age': 28, 'monthly_bill': 600}



# Exercise 24: Dataset Aggregation

## Background


Finance wants total revenue.


## Dataset / Input


Bills

300
500
700


## Task


Compute total bill.


## Hint


Convert to pandas.



In [ ]:
# Aggregate dataset to compute total bill
df_from_dataset = dataset.to_pandas()
total_bill = df_from_dataset["monthly_bill"].sum()
avg_bill = df_from_dataset["monthly_bill"].mean()

print("Dataset Aggregation:")
print(f"Total monthly bill: {total_bill}")
print(f"Average monthly bill: {avg_bill:.2f}")
print(f"Number of customers: {dataset.count()}")


2026-08-16 08:42:57,870	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_0_1 =======
2026-08-16 08:42:57,872	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-16 08:42:57,875	INFO logging_progress.py:227 -- Active & requested resources: 0/0 CPU, 0.0B/0.0B object store
2026-08-16 08:42:57,876	INFO logging_progress.py:192 -- ============================================
2026-08-16 08:42:57,906	INFO streaming_executor.py:327 -- ✔️  Dataset dataset_0_1 execution finished in 0.00 seconds


Dataset Aggregation:
Total monthly bill: 3000
Average monthly bill: 600.00
Number of customers: 5



# Exercise 25: Lazy Execution

## Background


Ray Data transformations are lazy.

Execution occurs only when the
dataset is consumed.


## Dataset / Input


Dataset


## Task


Create a transformation and
materialize it.


## Hint


Use show().



In [ ]:
# Lazy execution - transformations don't execute until materialized
lazy_dataset = (dataset
                .map(add_annual_bill)
                .filter(lambda row: row["monthly_bill"] > 400))

print("Lazy transformation created (not executed yet)")

# Materialize the dataset with show()
print("\nMaterialized dataset:")
lazy_dataset.show()

# Convert to pandas to inspect
print("\nAs pandas DataFrame:")
print(lazy_dataset.to_pandas())


Lazy transformation created (not executed yet)

Materialized dataset:


2026-08-16 08:42:57,956	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_7_0. Full logs are in /tmp/ray/session_2026-08-16_08-42-25_900245_1674369/logs/ray-data
2026-08-16 08:42:57,958	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[Map(add_annual_bill)->Filter(<lambda>)] -> LimitOperator[limit=20]
2026-08-16 08:42:57,991	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[Map(add_annual_bill)->Filter(<lambda>)]. The job may hang forever unless the cluster scales up.
2026-08-16 08:42:58,013	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_7_0 =======
2026-08-16 08:42:58,014	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-16 08:42:58,016	INFO logging_progress.py:227 -- Active & requested resources: 0/0 CPU, 0.0B/0.0B object store
2026-08-16 08:42:58,017	INFO logging_progress.py:181 -- 
2026-08-16 08:42:58,018	I

{'customer': 'B', 'age': 30, 'monthly_bill': 500, 'annual_bill': 6000}
{'customer': 'C', 'age': 35, 'monthly_bill': 700, 'annual_bill': 8400}
{'customer': 'D', 'age': 40, 'monthly_bill': 900, 'annual_bill': 10800}
{'customer': 'E', 'age': 28, 'monthly_bill': 600, 'annual_bill': 7200}

As pandas DataFrame:


2026-08-16 08:42:58,171	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[Map(add_annual_bill)->Filter(<lambda>)]. The job may hang forever unless the cluster scales up.
2026-08-16 08:42:58,189	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_0 =======
2026-08-16 08:42:58,190	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-16 08:42:58,191	INFO logging_progress.py:227 -- Active & requested resources: 0/0 CPU, 0.0B/0.0B object store
2026-08-16 08:42:58,193	INFO logging_progress.py:181 -- 
2026-08-16 08:42:58,194	INFO logging_progress.py:231 -- Map(add_annual_bill)->Filter(<lambda>): 0/1
2026-08-16 08:42:58,195	INFO logging_progress.py:233 --   Tasks: 1 [backpressured:tasks(ResourceBudget)]; Actors: 0; Queued blocks: 4 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-08-16 08:42:58,195	INFO logging_progress.py:192 -- ============================================
2026-08-16 08:42:58,318	INFO streaming

  customer  age  monthly_bill  annual_bill
0        B   30           500         6000
1        C   35           700         8400
2        D   40           900        10800
3        E   28           600         7200



# Ray Architecture

Driver
Workers
Actors
Object Store
Schedulers



# Exercise 26: Driver and Worker Concept

## Background


Ray applications consist of a Driver
and multiple Workers.


## Dataset / Input


No Dataset


## Task


Explain Driver and Worker roles.


## Hint


Think about task submission.



In [ ]:
# Driver and Worker concept explanation
explanation = """
Ray Architecture - Driver and Worker Roles:

DRIVER:
- The main Python process that initiates and coordinates tasks/actors
- Submits work to the Ray cluster
- Receives results from workers
- Runs the user's code (the main script/notebook)
- One driver per application

WORKERS:
- Execute the actual tasks and actor methods
- Receive work from the scheduler
- Process remote functions and actor calls
- Return results to the driver
- Multiple workers run in parallel on different cores/nodes
- Each worker maintains its own Python process

SCHEDULER:
- Distributes tasks to available workers
- Manages load balancing and resource allocation
- Ensures efficient task execution

OBJECT STORE:
- Stores intermediate results and data
- Used for passing data between tasks efficiently
- Reduces network overhead by sharing references instead of copying data
"""

print(explanation)



Ray Architecture - Driver and Worker Roles:

DRIVER:
- The main Python process that initiates and coordinates tasks/actors
- Submits work to the Ray cluster
- Receives results from workers
- Runs the user's code (the main script/notebook)
- One driver per application

WORKERS:
- Execute the actual tasks and actor methods
- Receive work from the scheduler
- Process remote functions and actor calls
- Return results to the driver
- Multiple workers run in parallel on different cores/nodes
- Each worker maintains its own Python process

SCHEDULER:
- Distributes tasks to available workers
- Manages load balancing and resource allocation
- Ensures efficient task execution

OBJECT STORE:
- Stores intermediate results and data
- Used for passing data between tasks efficiently
- Reduces network overhead by sharing references instead of copying data




# Exercise 27: Object References

## Background


Ray stores results as object references.


## Dataset / Input


Number = 10


## Task


Create and retrieve an object ref.


## Hint


Use ray.put().



In [ ]:
# Create and retrieve object references
number = 10

# Put object in object store
obj_ref = ray.put(number)
print(f"Object reference created: {obj_ref}")
print(f"Object type: {type(obj_ref)}")

# Retrieve object from object store
retrieved_value = ray.get(obj_ref)
print(f"Retrieved value: {retrieved_value}")
print(f"Value type: {type(retrieved_value)}")

# Object references can be passed to remote functions
@ray.remote
def multiply_by_2(obj_ref):
    value = ray.get(obj_ref)
    return value * 2

result = ray.get(multiply_by_2.remote(obj_ref))
print(f"Result of multiply_by_2: {result}")


Object reference created: ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000007e1f505)
Object type: <class 'ray.ObjectRef'>
Retrieved value: 10
Value type: <class 'int'>


RayTaskError(ValueError): [36mray::multiply_by_2()[39m (pid=1675283, ip=192.168.1.31)
  File "/tmp/ipykernel_1674369/1200531734.py", line 21, in multiply_by_2
    ...<2 lines>...
    )
ValueError: Invalid type of object refs, <class 'int'>, is given. 'object_refs' must either be an ObjectRef or a list of ObjectRefs.


# Ray Train

Ray Train enables distributed
model training.



# Exercise 28: Training Function

## Background


Distributed training begins with
a training function.


## Dataset / Input


No Dataset


## Task


Create train_func.


## Hint


Define a Python function.



In [ ]:
# =====================================================
# TODO
# =====================================================

# Create a training function
def train_func(config):
    """
    Simple training function that demonstrates distributed training.
    In a real scenario, this would train an actual model.
    """
    # Simulate training with given hyperparameters
    learning_rate = config.get("learning_rate", 0.01)
    batch_size = config.get("batch_size", 32)
    
    # Simulate training iterations
    accuracy = 0.5
    for epoch in range(5):
        # Simulate accuracy improvement
        accuracy += learning_rate * 0.1
    
    # Report results
    return {"accuracy": min(accuracy, 0.95), "learning_rate": learning_rate}

# Test the function
test_config = {"learning_rate": 0.01, "batch_size": 32}
result = train_func(test_config)
print("Training function created!")
print(f"Training result: {result}")



# Exercise 29: Scaling Configuration

## Background


Distributed training requires
resource allocation.


## Dataset / Input


No Dataset


## Task


Create ScalingConfig.


## Hint


Use ScalingConfig().



In [ ]:
# =====================================================
# TODO
# =====================================================

# Create ScalingConfig
from ray.train import ScalingConfig

scaling_config = ScalingConfig(
    num_workers=2,  # Number of training workers
    use_gpu=False,  # Don't use GPU for this example
    resources_per_worker={"CPU": 1}  # 1 CPU per worker
)

print("ScalingConfig created successfully!")
print(f"Number of workers: {scaling_config.num_workers}")
print(f"GPU: {scaling_config.use_gpu}")
print(f"Resources per worker: {scaling_config.resources_per_worker}")



# Exercise 30: Trainer Creation

## Background


A Trainer manages distributed
worker execution.


## Dataset / Input


No Dataset


## Task


Create a Trainer.


## Hint


Use TorchTrainer.



In [ ]:
# =====================================================
# TODO
# =====================================================

# Create a Trainer for distributed training
from ray.train import TorchTrainer

trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    scaling_config=scaling_config,
    run_config=None
)

print("TorchTrainer created successfully!")
print(f"Trainer: {trainer}")

# Note: In a real scenario, you would call trainer.fit() to execute training
# result = trainer.fit()



# Ray Tune

Ray Tune automates
hyperparameter tuning.



# Exercise 31: Grid Search

## Background


Data scientists want to test
multiple learning rates.


## Dataset / Input


Learning Rates


## Task


Create a grid search.


## Hint


Use tune.grid_search().



In [ ]:
# =====================================================
# TODO
# =====================================================

# Create a grid search configuration
from ray import tune

search_space = {
    "learning_rate": tune.grid_search([0.001, 0.01, 0.1]),
    "batch_size": 32
}

print("Grid search configuration created!")
print(f"Search space: {search_space}")
print("Learning rates to test: [0.001, 0.01, 0.1]")



# Exercise 32: Objective Function

## Background


Tune evaluates configurations
using an objective function.


## Dataset / Input


Parameters


## Task


Create objective().


## Hint


Return a score.



In [ ]:
# =====================================================
# TODO
# =====================================================

# Create objective function for tuning
def objective(config):
    """
    Objective function to optimize during hyperparameter tuning.
    Returns a metric that Ray Tune will optimize.
    """
    learning_rate = config.get("learning_rate", 0.01)
    batch_size = config.get("batch_size", 32)
    
    # Simulate model training
    accuracy = 0.5
    for epoch in range(3):
        accuracy += learning_rate * 0.15  # Better learning rate = faster improvement
    
    # Return metric for optimization
    return {"accuracy": min(accuracy, 0.95)}

# Test the objective function
test_result = objective({"learning_rate": 0.01, "batch_size": 32})
print("Objective function created!")
print(f"Sample result: {test_result}")



# Exercise 33: Run Tuner

## Background


Ray Tune executes trials
and selects the best result.


## Dataset / Input


Search Space


## Task


Run a tuner.


## Hint


Use tune.Tuner().



In [ ]:
# =====================================================
# TODO
# =====================================================

# Run a tuner to find best hyperparameters
tuner = tune.Tuner(
    objective,
    param_space=search_space,
    tune_config=tune.TuneConfig(
        num_samples=1,  # Run each configuration once
    )
)

print("Tuner created successfully!")
print("Running hyperparameter search...")

# Execute the tuner
results = tuner.fit()

# Get the best result
best_result = results.get_best_result(metric="accuracy", mode="max")
print(f"\nBest result:")
print(f"Best accuracy: {best_result.metrics['accuracy']:.4f}")
print(f"Best config: {best_result.config}")



# Ray Serve

Ray Serve provides scalable
model serving.



# Exercise 34: Simple Deployment

## Background


An ML team wants to deploy
an inference service.


## Dataset / Input


No Dataset


## Task


Create a deployment.


## Hint


Use @serve.deployment.



In [ ]:
# =====================================================
# TODO
# =====================================================

# Create a deployment with Ray Serve
from ray import serve

# Shutdown previous Serve instance if any
try:
    serve.shutdown()
except:
    pass

# Start Serve
serve.start(detached=True)

# Define a deployment
@serve.deployment
class PricingPredictor:
    def __init__(self):
        self.model_version = "v1"
    
    def __call__(self, monthly_bill: float):
        """Predict annual bill based on monthly bill"""
        annual_bill = monthly_bill * 12
        return {"monthly_bill": monthly_bill, "annual_bill": annual_bill, "model": self.model_version}

# Deploy the model
serve.run(PricingPredictor.bind(), name="pricing_predictor", route_prefix="/predict")

print("Deployment created and running!")
print("Deployment available at: http://localhost:8000/predict")



# Exercise 35: Inference Request

## Background


Clients send requests to
deployed models.


## Dataset / Input


Input Data


## Task


Simulate inference.


## Hint


Call prediction logic.



In [ ]:
# =====================================================
# TODO
# =====================================================

# Simulate inference request
import requests

# Send request to the deployed model
try:
    response = requests.post("http://localhost:8000/predict", json=300)
    prediction = response.json()
    print("Inference request successful!")
    print(f"Response: {prediction}")
except Exception as e:
    print(f"Note: Serve deployment might not be accessible in notebook: {e}")
    
    # Alternative: Direct call
    print("\nDirect call to predictor:")
    predictor = PricingPredictor()
    result = predictor(monthly_bill=300)
    print(f"Prediction result: {result}")



# Exercise 36: Mini Project: Distributed Customer Analytics

## Background


You are working as a Machine
Learning Engineer.

Management wants customer
revenue analytics.

Build a distributed Ray workflow.


## Dataset / Input


customers =

[
    {"customer":"A","bill":300},
    {"customer":"B","bill":500},
    {"customer":"C","bill":700},
    {"customer":"D","bill":900}
]


## Task


Create a Ray Dataset.

1. Load data
2. Create annual_bill
3. Filter annual_bill > 6000
4. Display result


## Hint


Use Ray Data map()
and filter().



In [ ]:
# =====================================================
# TODO
# =====================================================

# Mini Project: Distributed Customer Analytics

# 1. Load data
customers = [
    {"customer": "A", "bill": 300},
    {"customer": "B", "bill": 500},
    {"customer": "C", "bill": 700},
    {"customer": "D", "bill": 900}
]

# 2. Create Ray Dataset
analytics_dataset = ray.data.from_items(customers)
print("Step 1-2: Ray Dataset created")

# 3. Create annual_bill feature
def create_annual_bill(row):
    row["annual_bill"] = row["bill"] * 12
    return row

analytics_dataset = analytics_dataset.map(create_annual_bill)
print("Step 3: Annual bill feature created")

# 4. Filter annual_bill > 6000
def high_revenue_filter(row):
    return row["annual_bill"] > 6000

high_revenue = analytics_dataset.filter(high_revenue_filter)

print("Step 4: Filtered for annual_bill > 6000")
print("\nFinal Result:")
high_revenue.show()

# Summary statistics
print("\nSummary:")
print(f"Total customers: {analytics_dataset.count()}")
print(f"High revenue customers: {high_revenue.count()}")



# Exercise 37: Mini Project: Hyperparameter Search

## Background


You are training a churn
prediction model.

Management wants the best
hyperparameters.


## Dataset / Input


Search Space

learning_rate

0.001
0.01
0.1


## Task


Perform a Ray Tune search
and identify the best result.


## Hint


Use grid_search().



In [ ]:
# =====================================================
# TODO
# =====================================================

# Mini Project: Hyperparameter Search for Churn Prediction

# Define objective function for churn prediction
def churn_objective(config):
    """
    Objective function to tune churn prediction model.
    Higher learning rate might lead to better accuracy (up to a point).
    """
    learning_rate = config.get("learning_rate", 0.001)
    
    # Simulate model training with different learning rates
    # In reality, you would train an actual model here
    base_accuracy = 0.70
    
    # Learning rate relationship with accuracy
    if learning_rate == 0.001:
        accuracy = 0.75
    elif learning_rate == 0.01:
        accuracy = 0.82
    elif learning_rate == 0.1:
        accuracy = 0.78  # Too high - overfitting
    else:
        accuracy = 0.70
    
    return {"accuracy": accuracy}

# Create search space
search_config = {
    "learning_rate": tune.grid_search([0.001, 0.01, 0.1])
}

print("Performing Ray Tune hyperparameter search...")

# Run the tuner
churn_tuner = tune.Tuner(
    churn_objective,
    param_space=search_config,
    tune_config=tune.TuneConfig(
        num_samples=1,
    )
)

# Get results
churn_results = churn_tuner.fit()

# Find best configuration
best_churn = churn_results.get_best_result(metric="accuracy", mode="max")

print("\n" + "="*50)
print("Hyperparameter Tuning Results")
print("="*50)
print(f"Best Learning Rate: {best_churn.config['learning_rate']}")
print(f"Best Accuracy: {best_churn.metrics['accuracy']:.4f}")
print("="*50)
